# Empirical evaluation of weight quantisation

For text-only models, quantise & look at "(nucleus?) KL divergence" and perplexity on WikiText-2.

 - Which formats have the best compression ratio tradeoff?
 - How much does shuffling grouped elements hurt?
 - Does channel-with-sparse perform competitively with group-quantised?

In [6]:
%load_ext autoreload
%autoreload 2

import datasets
from functools import partial
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats
import seaborn as sns
import tqdm
import torch
from torch import tensor, Tensor
import transformers
from typing import Any, Literal

import quantisation.quantisation as Q
import format_experiments as E

matplotlib.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})

DEFAULT_DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [148]:
import gc
torch.cuda.empty_cache()
gc.collect()

4625

In [ ]:
rmodel = E.RequantisableModel.load("meta-llama/Llama-3.2-3B-Instruct", device=DEFAULT_DEVICE, dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [124]:
rmodel.reset()
data = E.Dataset.load_wikitext2(rmodel.model, 4096, 1, 100, DEFAULT_DEVICE, token_limit=10 * (4096 - 1))

In [116]:
fmt, compile = Q.LinearScalingFormatV2(Q.parse("E2M1"), Q.BFLOAT16, (1, 256), "absmax"), False
# fmt, compile = Q.LinearScalingFormatV2(Q.NF4, Q.BFLOAT16, (1, 256), "absmax"), True
# fmt, compile = Q.LinearScalingFormatV2(Q.crp_trunc_gauss(4, 256, True), Q.BFLOAT16, (1, 256), "absmax"), True
# fmt, compile = Q.LinearScalingFormatV2(Q.crp_trunc_laplace(4, 256, True), Q.BFLOAT16, (1, 256), "absmax"), True
# fmt, compile = Q.LinearScalingFormatV2(Q.crp_trunc_gauss(4, 256, False), Q.BFLOAT16, (1, 256), "signmax"), True
# fmt, compile = Q.LinearScalingFormatV2(Q.crp_trunc_laplace(4, 256, False), Q.BFLOAT16, (1, 256), "signmax"), True
log = rmodel.quantise(fmt, compile=compile)
E.evaluate_model(data, rmodel.model)

{'cross_entropy': 2.3427844047546387, 'kl_div': 0.08580692857503891}

In [106]:
pd.DataFrame.from_records(log).sort_values("snr").dropna()

,name,quantised,rmse,snr,bits
236,model.layers.26.self_attn.k_proj.weight,True,0.002940,68.0,12779520.0
245,model.layers.27.self_attn.k_proj.weight,True,0.002814,75.5,12779520.0
2,model.layers.0.self_attn.k_proj.weight,True,0.004713,77.0,12779520.0
1,model.layers.0.self_attn.q_proj.weight,True,0.003801,80.5,38338560.0
218,model.layers.24.self_attn.k_proj.weight,True,0.002971,82.0,12779520.0
...,...,...,...,...,...
213,model.layers.23.mlp.up_proj.weight,True,0.001829,101.5,102236160.0
222,model.layers.24.mlp.up_proj.weight,True,0.001829,101.5,102236160.0
204,model.layers.22.mlp.up_proj.weight,True,0.001833,101.5,102236160.0
184,model.layers.20.self_attn.o_proj.weight,True,0.001637,101.5,38338560.0
